# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/falah-bit/flyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task

**Lane:** Content Refresh Prioritization
**Task type:** Classification (binary)

I'm framing this problem as binary classification: predicting whether a page's
performance is currently **declining** (`is_declining_label`) based on its content
properties and 90-day activity signals — WITHOUT looking at the trend itself. The
classification output is then ranked to produce a prioritized review queue for the
content team.

In [15]:
import pandas as pd

!git clone https://github.com/falah-bit/flyrank-ml-internship.git
%cd flyrank-ml-internship/work/notebooks

df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")
print(df.shape)
df.head()

Cloning into 'flyrank-ml-internship'...
remote: Enumerating objects: 132, done.
remote: Counting objects: 100% (132/132), done.
remote: Compressing objects: 100% (88/88), done.
remote: Total 132 (delta 43), reused 93 (delta 28), pack-reused 0 (from 0)
Receiving objects: 100% (132/132), 1.88 MiB | 17.01 MiB/s, done.
Resolving deltas: 100% (43/43), done.
/content/flyrank-ml-internship/work/notebooks/flyrank-ml-internship/work/notebooks/flyrank-ml-internship/work/notebooks
(30000, 44)


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


### Quick exploration before defining the target & features

In [16]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 30000 entries, 0 to 29999
Data columns (total 44 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   content_id              30000 non-null  object 
 1   client_id               30000 non-null  object 
 2   search_volume           27532 non-null  float64
 3   competition             27532 non-null  float64
 4   competition_level       27390 non-null  object 
 5   cpc                     27532 non-null  float64
 6   content_type            30000 non-null  object 
 7   main_intent             27626 non-null  object 
 8   word_count              22301 non-null  float64
 9   char_count              22301 non-null  float64
 10  provider_used           8562 non-null   object 
 11  model_used              24267 non-null  object 
 12  impressions_90d         30000 non-null  int64  
 13  clicks_90d              30000 non-null  int64  
 14  pageviews_90d           30000 non-null

In [17]:
missing = df.isna().sum().sort_values(ascending=False)
missing[missing > 0]

,0
provider_used,21438
word_count,7699
char_count,7699
word_count_tier,7699
char_count_tier,7699
model_used,5733
trend_pct,3388
competition_level,2610
search_volume,2468
cpc,2468


In [18]:
# The data dictionary notes that keyword-context columns (search_volume, competition, etc.)
# are missing along content_type lines (e.g. 'feedly article' rows). Verify this:
pd.crosstab(df["content_type"], df["search_volume"].isna())

search_volume,False,True
content_type,,
comparison article,697,0
feedly article,0,2096
keyword article,26835,372


In [19]:
df["content_type"].value_counts()

,count
content_type,
keyword article,27207
feedly article,2096
comparison article,697


In [20]:
df["main_intent"].value_counts(dropna=False)

,count
main_intent,
informational,17235
transactional,5733
commercial,4612
NaN,2374
navigational,46


In [21]:
df["client_id"].nunique(), df.groupby("client_id").size().describe()

(32,
 count      32.000000
 mean      937.500000
 std      1376.387113
 min         3.000000
 25%       110.250000
 50%       567.000000
 75%      1058.750000
 max      7008.000000
 dtype: float64)

In [22]:
df["trend_direction"].value_counts(normalize=True)

,proportion
trend_direction,
down,0.542067
stable,0.198733
up,0.146267
new,0.074533
flat,0.038400


**Exploration notes:**
- Missingness in the keyword-context columns (search_volume, competition, cpc) is
  systematic along `content_type` — not random — so a blind `fillna(0)` would silently
  encode content type into the features.
- `client_id` has 32 distinct values with varying row counts per client — this matters
  for the train/test split later (client-holdout split, not a plain random split) to
  avoid leaking information across the split from the same client.
- `trend_direction` is confirmed as the sole source of the label; its proportions line
  up with the `is_declining_label` I compute in Section 2.

## 2. Target or proxy

Target: **`is_declining_label`** — officially defined by the pipeline as
`trend_direction == "down"` (1 = declining, 0 = not).

`trend_direction` and `trend_pct` **must never be used as features**, since they are the
source of the label (direct leakage). I'm also treating `impressions_last_30d` /
`clicks_last_30d` / `sessions_last_30d` and their `*_prev_30d` counterparts with caution,
since they're the raw inputs used to compute the trend — a potential indirect leakage
risk.

In [23]:
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)
df["is_declining_label"].value_counts(normalize=True)

,proportion
is_declining_label,
1,0.542067
0,0.457933


## 3. Success metric

**Primary metric: Precision@50** (matches the content team's limited weekly review
capacity — only the top-ranked pages will actually get acted on).

**Supporting metric: ROC-AUC / F1-score** to evaluate overall model quality, since the
classes in this dataset are fairly balanced (~54% vs ~46%), unlike my initial assumption
that the data would be heavily imbalanced.

In [24]:
df["is_declining_label"].value_counts()

,count
is_declining_label,
1,16262
0,13738


## 4. The unit of analysis, as a real dataframe

**One row = one content item (page) belonging to one client**, aggregated over the
trailing 90 days as of the export time. Identified by `content_id` (with `client_id`
used for grouping/splits only, never as a feature).

In [25]:
unit_cols = [
    "content_id", "client_id", "content_type", "main_intent",
    "word_count", "content_age_days", "days_since_last_update",
    "impressions_90d", "clicks_90d", "ctr", "avg_position",
    "is_declining_label",
]
df[unit_cols].head(10)

,content_id,client_id,content_type,main_intent,word_count,content_age_days,days_since_last_update,impressions_90d,clicks_90d,ctr,avg_position,is_declining_label
0,content_304f48230142,client_f369cb89fc,keyword article,transactional,3221.0,187,20,3803,29,0.76,10.6,1
1,content_a1fb4e703a9e,client_4e07408562,keyword article,informational,2481.0,445,25,15320,7,0.05,20.3,1
2,content_9aa793d4d895,client_7f2253d7e2,keyword article,informational,3515.0,141,20,12581,11,0.09,36.5,1
3,content_331d6c4de07b,client_19581e27de,keyword article,commercial,NaN,463,22,11751,58,0.49,6.2,0
4,content_d99b7a2d90ca,client_3fdba35f04,keyword article,informational,2803.0,263,14,19140,24,0.13,44.0,1
5,content_d4084a4bc775,client_f369cb89fc,keyword article,transactional,3080.0,147,20,3970,1,0.03,8.5,1
6,content_9a34b442b552,client_8722616204,keyword article,informational,3059.0,90,20,20,0,0.00,7.0,1
7,content_a63219c6e95a,client_19581e27de,keyword article,commercial,NaN,445,22,1724,1,0.06,21.2,0
8,content_5e6c160719bc,client_6208ef0f77,keyword article,informational,3807.0,90,20,32574,29,0.09,46.0,1
9,content_c27558df2b0c,client_19581e27de,keyword article,informational,NaN,257,104,1240,2,0.16,4.9,1


## 5. Why ML beats a fixed rule here

A simple rule (e.g. "flag pages with `avg_position` > 20") fails to capture the
interaction between many signals at once: `content_type`, `main_intent`, `word_count`,
`content_age_days`, `ctr`, `engagement_rate`, etc. — each carries different weight
depending on the combination. An ML model can learn these weights and interactions from
historical data. This repo's reference pipeline demonstrates it directly: the rule-based
baseline only reaches Precision@50 ≈ 0.24, while the model (logistic regression /
decision tree / random forest) reaches ≈ 0.68–0.74 — roughly a 3x lift.

In [26]:
# Try one simple rule: "declining" if avg_position is above the median
simple_rule = (df["avg_position"] > df["avg_position"].median()).astype(int)

# Compare this rule against the actual label
from sklearn.metrics import precision_score, recall_score, accuracy_score

print("Simple rule (avg_position > median) vs is_declining_label:")
print("Accuracy :", accuracy_score(df["is_declining_label"], simple_rule))
print("Precision:", precision_score(df["is_declining_label"], simple_rule))
print("Recall   :", recall_score(df["is_declining_label"], simple_rule))

Simple rule (avg_position > median) vs is_declining_label:
Accuracy : 0.5213666666666666
Precision: 0.5637606379414327
Recall   : 0.5173410404624278


In [27]:
# Try a rule combining 2 signals - shows how quickly this gets messy once signals interact
combo_rule = (
    (df["avg_position"] > df["avg_position"].median()) &
    (df["ctr"] < df["ctr"].median())
).astype(int)

print("Combined rule (2 columns) vs is_declining_label:")
print("Accuracy :", accuracy_score(df["is_declining_label"], combo_rule))
print("Precision:", precision_score(df["is_declining_label"], combo_rule))
print("Recall   :", recall_score(df["is_declining_label"], combo_rule))

Combined rule (2 columns) vs is_declining_label:
Accuracy : 0.4891
Precision: 0.5575951706295429
Recall   : 0.2783175501168368


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.